# Final Architecture Study — Phase 4 (FP32 & QAT INT8)

**Author:** Rafael
**Dataset:** Tiny-ImageNet-200 (200 classes, 64x64 RGB)
**Goal:** Combine Phase 3's most successful compensation mechanisms (Bottleneck,
Fire, Residual, Depthwise-Separable) into four hybrid architectures, train each
in FP32, QAT fine-tune, convert to INT8, and compare against every prior phase.

| Model | Mechanisms combined | QAT |
|---|---|---|
| AlexNetFinalBottleneckFire | Bottleneck stem + Fire body | Yes |
| AlexNetFinalFireResidual | Fire + residual shortcuts | Yes |
| AlexNetFinalBottleneckResidual | Bottleneck + residual shortcuts | Yes |
| AlexNetFinalDepthwiseFire | Depthwise-separable stem + Fire body | Yes |

## Notebook layout

1. Imports & reproducibility
2. Dataset & loaders
3. Model shape check
4. Model registration (fuse maps via `find_fuse_groups`)
5. FP32 training
6. QAT training
7. INT8 conversion & CPU evaluation
8. FP32 evaluation (reload best checkpoints)
9. Final comparison table + benchmarks + per-model summaries
10. Ablation study (component contribution)
11. Cross-phase comparison (Phases 1-4)
12. Persist experiment summary

## 1. Imports & reproducibility

In [ ]:
import json
import os
import random
import sys
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchinfo
import wandb

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    MODEL_REGISTRY, register_model,
    Trainer, make_qat_callback,
    build_qat, build_comparison_table,
    load_best_model, convert_to_int8,
    find_fuse_groups,
    auto_resume_path, compress_checkpoint,
    create_results_summary, disk_mb, gzip_mb,
    compute_flops, make_run_summary,
)
from configs.loader import load_config

from models import (
    AlexNetFinalBottleneckFire,
    AlexNetFinalFireResidual,
    AlexNetFinalBottleneckResidual,
    AlexNetFinalDepthwiseFire,
)

torch.backends.quantized.engine = "fbgemm"

In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR    = project_root / "checkpoints" / "final_architecture_phase4"
RESULTS_DIR = project_root / "results" / "phase_4_compression_and_final_architecture_training"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data_cfg = DataConfig(**load_config("data.yaml"))
data_cfg.seed = SEED
fp32_cfg = TrainerConfig(**load_config("training.yaml"))
qat_cfg  = QATConfig(**load_config("qat.yaml"))
print(device)

cuda


## 2. Dataset & loaders

In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path
print("Tiny-ImageNet train path:", train_path)

train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg)

print(f"Train samples: {len(train_ds):,}")
print(f"Val   samples: {len(val_ds):,}")
print(f"Classes:       {data_cfg.num_classes}")
print(f"Batches/epoch: train={len(train_loader)}, val={len(val_loader)}")

Tiny-ImageNet train path: /home/rafael/.cache/kagglehub/datasets/akash2sharma/tiny-imagenet/versions/1/tiny-imagenet-200/train
Train samples: 90,000
Val   samples: 10,000
Classes:       200
Batches/epoch: train=1406, val=157


## 3. Model shape check

Four Phase 4 hybrid architectures — all from `models/final_architecture.py`:

| Name | Mechanisms | Head |
|---|---|---|
| AlexNetFinalBottleneckFire | Bottleneck stem + Fire body | Linear(256, 200) |
| AlexNetFinalFireResidual | Fire + residual shortcuts | Linear(256, 200) |
| AlexNetFinalBottleneckResidual | Bottleneck + residual shortcuts | Linear(256, 200) |
| AlexNetFinalDepthwiseFire | Depthwise-separable stem + Fire body | Linear(256, 200) |

All use GAP heads (Phase 3's smallest, most quantization-stable pattern) and are
fully QAT-compatible — no Sigmoid, all skip-adds via `FloatFunctional`.

In [4]:
x = torch.randn(2, 3, data_cfg.img_size, data_cfg.img_size)
for label, ctor in [
    ("AlexNetFinalBottleneckFire",     AlexNetFinalBottleneckFire),
    ("AlexNetFinalFireResidual",       AlexNetFinalFireResidual),
    ("AlexNetFinalBottleneckResidual", AlexNetFinalBottleneckResidual),
    ("AlexNetFinalDepthwiseFire",      AlexNetFinalDepthwiseFire),
]:
    m = ctor().eval()
    with torch.no_grad():
        y = m(x)
    assert y.shape == (2, data_cfg.num_classes), f"{label}: unexpected shape {y.shape}"
    params = sum(p.numel() for p in m.parameters())
    print(f"{label:32s} | params={params/1e6:.3f}M")

AlexNetFinalBottleneckFire       | params=0.506M
AlexNetFinalFireResidual         | params=0.699M
AlexNetFinalBottleneckResidual   | params=0.571M
AlexNetFinalDepthwiseFire        | params=0.475M


## 4. Model registration

All four models use nested block classes (`_AlexBottleneck`, `_FireModule`,
`_FireResBlock`, `_BottleneckResBlock`) — fuse maps are auto-detected via
`find_fuse_groups()`, same approach used for Phase 3's Bottleneck/Residual/Fire.

In [5]:
FUSE_BOTTLENECK_FIRE     = find_fuse_groups(AlexNetFinalBottleneckFire())
FUSE_FIRE_RESIDUAL       = find_fuse_groups(AlexNetFinalFireResidual())
FUSE_BOTTLENECK_RESIDUAL = find_fuse_groups(AlexNetFinalBottleneckResidual())
FUSE_DEPTHWISE_FIRE      = find_fuse_groups(AlexNetFinalDepthwiseFire())

MODEL_REGISTRY.clear()
register_model("alexnet_final_bottleneck_fire",     AlexNetFinalBottleneckFire,     fuse_map=FUSE_BOTTLENECK_FIRE,     lr=1e-3)
register_model("alexnet_final_fire_residual",       AlexNetFinalFireResidual,       fuse_map=FUSE_FIRE_RESIDUAL,       lr=1e-3)
register_model("alexnet_final_bottleneck_residual", AlexNetFinalBottleneckResidual, fuse_map=FUSE_BOTTLENECK_RESIDUAL, lr=1e-3)
register_model("alexnet_final_depthwise_fire",      AlexNetFinalDepthwiseFire,      fuse_map=FUSE_DEPTHWISE_FIRE,      lr=1e-3)

print("Registered:", list(MODEL_REGISTRY))

Registered: ['alexnet_final_bottleneck_fire', 'alexnet_final_fire_residual', 'alexnet_final_bottleneck_residual', 'alexnet_final_depthwise_fire']


## 5. FP32 training

In [6]:
fp32_training_results = {}

for name, spec in MODEL_REGISTRY.items():
    resume_from = auto_resume_path(SAVE_DIR, name)

    run_id = None
    if resume_from:
        meta_path = SAVE_DIR / f"{name}_meta.json"
        if meta_path.exists():
            run_id = json.loads(meta_path.read_text()).get("wandb_run_id")

    model_cfg = replace(fp32_cfg, lr=spec.get("lr", fp32_cfg.lr))
    print("=" * 72)
    print(f"Training: {name}  lr={model_cfg.lr}  epochs={model_cfg.epochs}")
    if resume_from:
        print("(Resuming from checkpoint)")
    print("=" * 72)

    model = spec["ctor"]().to(device)

    run = wandb.init(
        project="alexnet-phase4",
        group="fp32-phase4",
        name=f"{name}_fp32",
        id=run_id,
        resume="allow" if run_id else None,
        config={**asdict(model_cfg), "arch": name, "phase": "fp32",
                "num_classes": data_cfg.num_classes, "img_size": data_cfg.img_size,
                "dataset": "tiny-imagenet-200"},
        tags=["phase4", "final-architecture", "tiny-imagenet", "fp32"],
        mode="offline",
    )

    trainer = Trainer(
        model, train_loader, val_loader, model_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
        wandb_run=run,
        log_file=SAVE_DIR / f"{name}.log",
    )
    results = trainer.fit(resume_from=resume_from)
    wandb.finish()

    fp32_training_results[name] = results

    del model
    torch.cuda.empty_cache()

print("\nFP32 training complete.")

Training: alexnet_final_bottleneck_fire  lr=0.001  epochs=100
(Resuming from checkpoint)


wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id 2rlfwz6x.


Validation: 100%|██████████| 157/157 [00:01<00:00, 114.03it/s, loss=4.2320, top1=13.92%, top5=35.15%]
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   3/100 | train_loss=4.4332 train_acc=10.54% | val_loss=4.2320 val_acc=13.92% val_top5=35.15% | lr=9.98e-04 peak_mem=147MB time=24.2s
Validation: 100%|██████████| 157/157 [00:01<00:00, 151.44it/s, loss=3.9782, top1=18.84%, top5=42.88%]
Epoch   4/100 | train_loss=4.2798 train_acc=13.22% | val_loss=3.9782 val_acc=18.84% val_top5=42.88% | lr=9.96e-04 peak_mem=147MB time=21.6s
Validation: 100%|██████████| 157/157 [00:01<00:00, 146.71it/s, loss=3.8699, top1=21.45%, top5=47.00%]
Epoch   5/100 | train_loss=4.1757 train_acc=15.20% | val_loss=3.8699 val_acc=21.45% val_top5=47.00% | lr=9.94e-04 peak_mem=147MB time=20.7s
Va

best_val_acc,▁▂▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇▇▇████
epoch_time_s,▇▃▁▂▃▁▁▄▃▇▅▄▃▄▃▃▅█▆▄▄▅▃▃▃▃▄▄▁▁▁▁▂▁▄▂▃▃
lr,████████▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▂▂▂▂▁▁
peak_gpu_mem_mb,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇████████
train_loss,█▇▇▆▆▅▅▅▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_acc,▁▂▃▃▄▄▄▅▅▅▆▆▆▆▆▇▇▇▆▇▇▇▇▇▇▇█▇▇█████████
val_loss,█▇▆▆▅▄▄▄▄▄▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁
val_top5,▁▃▃▄▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇█▇████████████
best_val_acc,42.32
epoch_time_s,21.83557


Training: alexnet_final_fire_residual  lr=0.001  epochs=100


...

best_val_acc,▁▂▃▃▄▄▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇████
epoch_time_s,█▃▂▁▂▂▁▃▆▄▃▄▄▅▂▅▅▅▅▃▃▇▅▄▃▂▇▄▅▃▁▂▃▁▄▃▃▁
lr,█████████▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▂▂▂▁▁
peak_gpu_mem_mb,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▂▃▃▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇████████
train_loss,█▇▆▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc,▁▂▃▃▄▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇████████████
val_loss,█▇▆▅▅▅▄▄▃▃▃▃▂▂▃▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_top5,▁▂▃▄▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇█████████████████
best_val_acc,45.12
epoch_time_s,22.50517


Training: alexnet_final_depthwise_fire  lr=0.001  epochs=100


Validation: 100%|██████████| 157/157 [00:01<00:00, 154.87it/s, loss=4.7133, top1=5.87%, top5=19.74%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/100 | train_loss=5.0146 train_acc=2.99% | val_loss=4.7133 val_acc=5.87% val_top5=19.74% | lr=1.00e-03 peak_mem=106MB time=21.3s
Validation: 100%|██████████| 157/157 [00:01<00:00, 147.10it/s, loss=4.4891, top1=9.53%, top5=27.38%]
Epoch   2/100 | train_loss=4.7165 train_acc=6.46% | val_loss=4.4891 val_acc=9.53% val_top5=27.38% | lr=9.99e-04 peak_mem=106MB time=21.5s
Validation: 100%|██████████| 157/157 [00:01<00:00, 140.63it/s, loss=4.2617, top1=13.92%, top5=34.95%]
Epoch   3/100 | train_loss=4.5181 train_acc=9.41% | val_loss=4.2617 val_acc=13.92% val_top5=34.95% | lr=9.98e-04 peak_mem=106MB time=21.5s
Validation: 100%|██████████| 157/157 [00:00<00:00, 158.48it/s, loss=4.0669, top1=17.09%, top5=41.10%]
Epoch   4/100 | train_loss=4.3775 train_acc=11.91% | val_loss=4.0669 val_acc=1

best_val_acc,▁▂▂▃▃▄▄▄▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█████████
epoch_time_s,▂▂▂▂▃▃▃▂▄▄▅▅▅█▄▂▃▃▃▃▂▃▃▃▂▁▁▂▂▁▁▂▂▁▂▂▂▂▁▂
lr,█████████▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁
peak_gpu_mem_mb,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▂▃▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████████
train_loss,█▇▆▆▅▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▂▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇█████████████████
val_loss,█▇▆▆▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_top5,▁▃▄▄▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇███████████████
best_val_acc,43.53
epoch_time_s,21.16813



FP32 training complete.


## 6. Quantization-Aware Training (QAT)

All four Phase 4 models are QAT-compatible (no Sigmoid, `FloatFunctional` adds).
QAT trains on GPU; `convert` and INT8 inference run on CPU (fbgemm).

In [7]:
qat_train_cfg = replace(
    fp32_cfg,
    epochs=qat_cfg.epochs,
    lr=qat_cfg.lr,
    weight_decay=qat_cfg.weight_decay,
    use_amp=False,  # AMP incompatible with fake-quant observers
)

qat_models = {}
qat_training_results = {}

for name in MODEL_REGISTRY:
    resume_from = auto_resume_path(SAVE_DIR, f"qat_{name}")

    run_id = None
    if resume_from:
        meta_path = SAVE_DIR / f"qat_{name}_meta.json"
        if meta_path.exists():
            run_id = json.loads(meta_path.read_text()).get("wandb_run_id")

    print("=" * 72)
    print(f"QAT fine-tuning: {name}")
    if resume_from:
        print("(Resuming from checkpoint)")
    print("=" * 72)

    qat_model = build_qat(name, save_dir=SAVE_DIR, device=device)
    cb = make_qat_callback(qat_cfg.freeze_bn_epoch, qat_cfg.disable_observer_epoch)

    run = wandb.init(
        project="alexnet-phase4",
        group="qat-phase4",
        name=f"{name}_qat",
        id=run_id,
        resume="allow" if run_id else None,
        config={
            **asdict(qat_train_cfg),
            "arch": name,
            "phase": "qat",
            "freeze_bn_epoch": qat_cfg.freeze_bn_epoch,
            "disable_observer_epoch": qat_cfg.disable_observer_epoch,
            "num_classes": data_cfg.num_classes, "img_size": data_cfg.img_size,
            "dataset": "tiny-imagenet-200",
        },
        tags=["phase4", "final-architecture", "tiny-imagenet", "qat"],
        mode="offline",
    )

    trainer = Trainer(
        qat_model, train_loader, val_loader, qat_train_cfg,
        device, SAVE_DIR, f"qat_{name}", num_classes=data_cfg.num_classes,
        wandb_run=run,
        epoch_callback=cb,
        log_file=SAVE_DIR / f"qat_{name}.log",
    )
    results = trainer.fit(resume_from=resume_from)
    wandb.finish()

    qat_training_results[name] = results
    qat_models[name] = qat_model.cpu()

    del qat_model
    torch.cuda.empty_cache()

print("\nQAT training complete.")

/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


QAT fine-tuning: alexnet_final_bottleneck_fire


Validation: 100%|██████████| 157/157 [00:01<00:00, 105.96it/s, loss=2.9903, top1=43.59%, top5=69.75%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/20 | train_loss=3.1132 train_acc=39.41% | val_loss=2.9903 val_acc=43.59% val_top5=69.75% | lr=9.94e-06 peak_mem=454MB time=30.0s
Validation: 100%|██████████| 157/157 [00:01<00:00, 105.60it/s, loss=2.9755, top1=43.95%, top5=69.98%]
Epoch   2/20 | train_loss=3.0828 train_acc=40.27% | val_loss=2.9755 val_acc=43.95% val_top5=69.98% | lr=9.76e-06 peak_mem=454MB time=29.7s
Validation: 100%|██████████| 157/157 [00:01<00:00, 105.86it/s, loss=2.9745, top1=43.93%, top5=69.96%]
Epoch   3/20 | train_loss=3.0671 train_acc=40.73% | val_loss=2.9745 val_acc=43.93% val_top5=69.96% | lr=9.46e-06 peak_mem=454MB time=29.9s
Validation: 100%|██████████| 157/157 [00:01<00:00, 106.94it/s, loss=2.9663, top1=44.21%, top5=70.36%]
Epoch   4/20 | train_loss=3.0109 train_acc=41.86% | val_loss=2.9663 val_ac

best_val_acc,▁▄▆▇▇▇█
epoch_time_s,█▅▆▃▄▃▄▁▃▄▅▃▂▁▂▅▅▅▄
lr,███▇▇▇▆▆▅▄▄▃▃▂▂▂▁▁▁
peak_gpu_mem_mb,███▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▄▆▇▇▇▇█▇████▇██▇█
train_loss,█▆▅▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁
val_acc,▁▄▄▆▅▅▅▇▅▇▇▆▆█▇▇█▇▆
val_loss,█▄▄▂▂▁▃▁▂▂▂▁▂▁▂▁▂▁▁
val_top5,▁▃▃▆▅▆▄▇▄▅▅█▅▆▆▆▆▇█
best_val_acc,44.46
epoch_time_s,29.54118


/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


...

/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


QAT fine-tuning: alexnet_final_depthwise_fire


Validation: 100%|██████████| 157/157 [00:01<00:00, 125.61it/s, loss=3.0076, top1=43.13%, top5=69.03%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch   1/20 | train_loss=3.1617 train_acc=38.53% | val_loss=3.0076 val_acc=43.13% val_top5=69.03% | lr=9.94e-06 peak_mem=276MB time=21.9s
Validation: 100%|██████████| 157/157 [00:01<00:00, 134.05it/s, loss=3.0125, top1=43.09%, top5=68.59%]
Epoch   2/20 | train_loss=3.1534 train_acc=38.57% | val_loss=3.0125 val_acc=43.09% val_top5=68.59% | lr=9.76e-06 peak_mem=276MB time=21.7s
Validation: 100%|██████████| 157/157 [00:01<00:00, 131.18it/s, loss=3.0067, top1=43.30%, top5=69.23%]
Epoch   3/20 | train_loss=3.1539 train_acc=38.91% | val_loss=3.0067 val_acc=43.30% val_top5=69.23% | lr=9.46e-06 peak_mem=276MB time=21.8s
Validation: 100%|██████████| 157/157 [00:01<00:00, 127.34it/s, loss=3.0040, top1=43.82%, top5=69.29%]
Epoch   4/20 | train_loss=3.1007 train_acc=40.13% | val_loss=3.0040 val_ac

best_val_acc,▁▃█
epoch_time_s,█▄▅▁▃▃▂▅▃
lr,██▇▆▆▅▄▂▁
peak_gpu_mem_mb,███▁▁▁▁▁▁
train_acc,▁▁▂▇▇██▇█
train_loss,█▇▇▂▂▁▁▁▁
val_acc,▁▁▃█▆▇▄▄▅
val_loss,▅█▄▃▁▂▄▃▁
val_top5,▅▁▇▇▇█▆▇▆
best_val_acc,43.82
epoch_time_s,21.66564



QAT training complete.


## 7. INT8 conversion & CPU evaluation

`torch.ao.quantization.convert` materialises true quantised ops and must run on CPU.

In [ ]:
val_loader_cpu = torch.utils.data.DataLoader(
    val_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False,
)
cpu_device = torch.device("cpu")

int8_models = {name: convert_to_int8(m.cpu().eval()) for name, m in qat_models.items()}

for name, m in int8_models.items():
    torch.save(m.state_dict(), SAVE_DIR / f"{name}.pth")
    compress_checkpoint(SAVE_DIR / f"{name}.pth")
print("INT8 conversion done.")

int8_metrics = {}
for name, m in int8_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(
        m.cpu().eval(), val_loader_cpu, val_loader_cpu, dummy_cfg,
        cpu_device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    metrics = trainer.evaluate(topk=(1, 5))
    int8_metrics[name] = metrics
    print(f"{name:32s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

## 8. FP32 evaluation — reload best checkpoints

In [9]:
CTORS = {
    "alexnet_final_bottleneck_fire":     AlexNetFinalBottleneckFire,
    "alexnet_final_fire_residual":       AlexNetFinalFireResidual,
    "alexnet_final_bottleneck_residual": AlexNetFinalBottleneckResidual,
    "alexnet_final_depthwise_fire":      AlexNetFinalDepthwiseFire,
}

fp32_best = {name: load_best_model(name, ctor, SAVE_DIR, device) for name, ctor in CTORS.items()}

fp32_metrics = {}
for name, m in fp32_best.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(
        m, train_loader, val_loader, dummy_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    metrics = trainer.evaluate(topk=(1, 5))
    fp32_metrics[name] = metrics
    print(f"{name:32s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

alexnet_final_bottleneck_fire    | loss=2.4907 | top1=42.29% | top5=68.54%
alexnet_final_fire_residual      | loss=2.1318 | top1=49.79% | top5=74.80%
alexnet_final_bottleneck_residual | loss=2.3388 | top1=45.10% | top5=71.15%
alexnet_final_depthwise_fire     | loss=2.4434 | top1=43.46% | top5=69.02%


## 9. Final comparison table, benchmarks & per-model summaries

In [10]:
rows = []

for name, m in fp32_best.items():
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": name, "precision": "FP32",
        "top1_%": fp32_metrics[name]["top1"],
        "top5_%": fp32_metrics[name]["top5"],
        "loss":   fp32_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}_best.pth"),
    })

for name, m in int8_models.items():
    info = torchinfo.summary(fp32_best[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": f"{name}_INT8", "precision": "INT8",
        "top1_%": int8_metrics[name]["top1"],
        "top5_%": int8_metrics[name]["top5"],
        "loss":   int8_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}.pth"),
    })

df = build_comparison_table(rows)
df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)
df

,model,precision,top1_%,top5_%,loss,params_M,size_MB
0,alexnet_final_fire_residual,FP32,49.793291,74.802184,2.131824,0.698664,8.093657
1,alexnet_final_bottleneck_residual,FP32,45.099601,71.151185,2.338797,0.571304,6.653960
2,alexnet_final_depthwise_fire,FP32,43.455875,69.023257,2.443448,0.474921,5.511263
3,alexnet_final_bottleneck_fire,FP32,42.286080,68.539512,2.490701,0.506408,5.882781
4,alexnet_final_fire_residual_INT8,INT8,49.201027,74.387586,2.153559,0.698664,0.746309
5,alexnet_final_bottleneck_residual_INT8,INT8,45.980674,72.328943,2.280574,0.571304,0.637690
6,alexnet_final_bottleneck_fire_INT8,INT8,44.001037,70.151949,2.381651,0.506408,0.545216
7,alexnet_final_depthwise_fire_INT8,INT8,42.789248,68.953073,2.452214,0.474921,0.509390


In [ ]:
# --- Benchmark FP32 models (GPU) ---
fp32_benchmarks = {}
for name, m in fp32_best.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    t = Trainer(m, train_loader, val_loader, dummy_cfg, device, SAVE_DIR, name,
                num_classes=data_cfg.num_classes)
    fp32_benchmarks[name] = t.benchmark()
    print(f"{name:32s} FP32 | {fp32_benchmarks[name]['latency_ms_per_image']:.3f} ms/img")

# --- Benchmark INT8 models (CPU) ---
int8_benchmarks = {}
for name, m in int8_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    t = Trainer(m.cpu().eval(), val_loader_cpu, val_loader_cpu, dummy_cfg, cpu_device, SAVE_DIR, name,
                num_classes=data_cfg.num_classes)
    int8_benchmarks[name] = t.benchmark(warmup=50)
    print(f"{name:32s} INT8 | {int8_benchmarks[name]['latency_ms_per_image']:.3f} ms/img")

# --- FLOPs ---
fp32_flops = {}
for name, m in fp32_best.items():
    fp32_flops[name] = compute_flops(m.cpu().eval())
    print(f"{name:32s} | {fp32_flops[name]['macs']/1e9:.3f} GMACs")

# --- Per-model summary JSONs ---
for name in fp32_best:
    compress_checkpoint(SAVE_DIR / f"{name}_best.pth")
    info = torchinfo.summary(fp32_best[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    summary = make_run_summary(
        name=name,
        mode="fp32+qat+int8",
        fit_results=fp32_training_results.get(name, {}),
        fp32_eval=fp32_metrics[name],
        params_m=info.total_params / 1e6,
        fp32_size_mb=disk_mb(SAVE_DIR / f"{name}_best.pth"),
        int8_size_mb=disk_mb(SAVE_DIR / f"{name}.pth"),
        fp32_benchmark=fp32_benchmarks[name],
        flops_results=fp32_flops[name],
        int8_eval=int8_metrics.get(name),
        int8_benchmark=int8_benchmarks.get(name),
        fp32_gzip_mb=gzip_mb(SAVE_DIR / f"{name}_best.pth"),
        int8_gzip_mb=gzip_mb(SAVE_DIR / f"{name}.pth"),
    )
    out = RESULTS_DIR / f"{name}_summary.json"
    with open(out, "w") as f:
        json.dump(summary, f, indent=2, default=str)
    print(f"Saved: {out.name}")

In [12]:
print("=" * 72)
print("RANKING BY TOP-1 ACCURACY (all precisions)")
print("=" * 72)
ranked = df.sort_values("top1_%", ascending=False).reset_index(drop=True)
for i, row in ranked.iterrows():
    print(f"{i+1:2d}. {row['model']:34s} [{row['precision']}] | "
          f"top1={row['top1_%']:6.2f}% | top5={row['top5_%']:6.2f}% | "
          f"params={row['params_M']:6.2f}M | size={row['size_MB']:6.2f}MB")

RANKING BY TOP-1 ACCURACY (all precisions)
 1. alexnet_final_fire_residual        [FP32] | top1= 49.79% | top5= 74.80% | params=  0.70M | size=  8.09MB
 2. alexnet_final_fire_residual_INT8   [INT8] | top1= 49.20% | top5= 74.39% | params=  0.70M | size=  0.75MB
 3. alexnet_final_bottleneck_residual_INT8 [INT8] | top1= 45.98% | top5= 72.33% | params=  0.57M | size=  0.64MB
 4. alexnet_final_bottleneck_residual  [FP32] | top1= 45.10% | top5= 71.15% | params=  0.57M | size=  6.65MB
 5. alexnet_final_bottleneck_fire_INT8 [INT8] | top1= 44.00% | top5= 70.15% | params=  0.51M | size=  0.55MB
 6. alexnet_final_depthwise_fire       [FP32] | top1= 43.46% | top5= 69.02% | params=  0.47M | size=  5.51MB
 7. alexnet_final_depthwise_fire_INT8  [INT8] | top1= 42.79% | top5= 68.95% | params=  0.47M | size=  0.51MB
 8. alexnet_final_bottleneck_fire      [FP32] | top1= 42.29% | top5= 68.54% | params=  0.51M | size=  5.88MB


## 10. Ablation study

Each Phase 4 model already isolates a *pair* of mechanisms. This section
compares each hybrid against its two single-mechanism Phase 3 parents (already
trained in `results/phase_3_compensation_and_hybrids_training/`) to see whether combining mechanisms
compounds or cancels their individual effects — no retraining needed.

In [13]:
PHASE3_DIR = project_root / "results" / "phase_3_compensation_and_hybrids_training"

ABLATION_PARENTS = {
    "alexnet_final_bottleneck_fire":     ["alexnet_bottleneck", "alexnet_fire"],
    "alexnet_final_fire_residual":       ["alexnet_fire", "alexnet_residual"],
    "alexnet_final_bottleneck_residual": ["alexnet_bottleneck", "alexnet_residual"],
    "alexnet_final_depthwise_fire":      ["alexnet_depthwisesep", "alexnet_fire"],
}

def _load_phase3_top1(name):
    p = PHASE3_DIR / f"{name}_summary.json"
    if not p.exists():
        return None
    return json.loads(p.read_text()).get("fp32_top1")

ablation_rows = []
for hybrid_name, parents in ABLATION_PARENTS.items():
    hybrid_top1 = fp32_metrics[hybrid_name]["top1"]
    parent_top1s = {p: _load_phase3_top1(p) for p in parents}
    best_parent_top1 = max((v for v in parent_top1s.values() if v is not None), default=None)
    ablation_rows.append({
        "hybrid": hybrid_name,
        "hybrid_top1": hybrid_top1,
        **{f"parent_{p}_top1": v for p, v in parent_top1s.items()},
        "best_parent_top1": best_parent_top1,
        "delta_vs_best_parent": (hybrid_top1 - best_parent_top1) if best_parent_top1 is not None else None,
    })

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(RESULTS_DIR / "ablation_results.csv", index=False)
ablation_df

,hybrid,hybrid_top1,parent_alexnet_bottleneck_top1,parent_alexnet_fire_top1,best_parent_top1,delta_vs_best_parent,parent_alexnet_residual_top1,parent_alexnet_depthwisesep_top1
0,alexnet_final_bottleneck_fire,42.286080,44.622979,43.976474,44.622979,-2.336898,NaN,NaN
1,alexnet_final_fire_residual,49.793291,NaN,43.976474,48.012939,1.780352,48.012939,NaN
2,alexnet_final_bottleneck_residual,45.099601,44.622979,NaN,48.012939,-2.913338,48.012939,NaN
3,alexnet_final_depthwise_fire,43.455875,NaN,43.976474,44.390011,-0.934136,NaN,44.390011


## 11. Cross-phase comparison

Aggregate every phase's per-model summary JSON (Phase 1 baselines, Phase 2
kernel restriction, Phase 3 compensation, Phase 4 final architectures) into a
single ranking.

In [14]:
phase_dirs = {
    "Phase 1": project_root / "results" / "phase_1_baseline_training",
    "Phase 2": project_root / "results" / "phase_2_kernel_restriction_training",
    "Phase 3": project_root / "results" / "phase_3_compensation_and_hybrids_training",
    "Phase 4": RESULTS_DIR,
}

all_rows = []
for phase_name, phase_dir in phase_dirs.items():
    if not phase_dir.exists():
        continue
    for summary_file in phase_dir.glob("*_summary.json"):
        data = json.loads(summary_file.read_text())
        data["phase"] = phase_name
        all_rows.append(data)

cross_phase_df = pd.DataFrame(all_rows)
cross_phase_cols = [c for c in [
    "phase", "model_name", "fp32_top1", "int8_top1", "params_m",
    "fp32_size_mb", "int8_size_mb", "param_efficiency_top1_per_m",
    "quantization_drop_top1",
] if c in cross_phase_df.columns]
cross_phase_display = cross_phase_df[cross_phase_cols].sort_values("fp32_top1", ascending=False)
cross_phase_display.to_csv(RESULTS_DIR / "cross_phase_comparison.csv", index=False)
cross_phase_display

,phase,model_name,fp32_top1,int8_top1,params_m,fp32_size_mb,int8_size_mb,param_efficiency_top1_per_m,quantization_drop_top1
2,Phase 1,mobilenetv2,57.995582,NaN,2.480072,28.751925,NaN,23.414643,NaN
1,Phase 1,resnet18_tv,53.913635,NaN,11.279112,129.208681,NaN,4.785838,NaN
4,Phase 1,vgg_style,51.814806,51.188743,2.405288,27.584812,2.376703,21.556670,0.626063
22,Phase 4,alexnet_final_fire_residual,49.793291,49.201027,0.698664,8.093657,0.746309,71.250272,0.592265
20,Phase 3,alexnet_residual,48.012939,47.273704,60.670280,694.414034,58.102711,0.792151,0.739235
8,Phase 2,alexnet_small_kernel,45.836565,35.946080,1.602376,18.353315,1.561041,28.607518,9.890485
24,Phase 4,alexnet_final_bottleneck_residual,45.099601,45.980674,0.571304,6.653960,0.637690,78.977217,-0.881073
12,Phase 3,alexnet_bottleneck,44.622979,44.539082,0.385000,4.491745,0.431780,116.051948,0.083897
6,Phase 2,alexnet_stacked,44.559684,42.792845,60.483976,692.254669,57.941553,0.737881,1.766840
16,Phase 3,alexnet_depthwisesep,44.390011,41.469061,0.313641,3.654016,0.399755,141.818193,2.920949


## 12. Persist experiment summary

In [15]:
create_results_summary(
    results={
        "fp32_metrics": fp32_metrics,
        "int8_metrics": int8_metrics,
        "fp32_training_results": fp32_training_results,
        "qat_training_results": qat_training_results,
        "ablation": ablation_rows,
    },
    config=asdict(fp32_cfg),
    output_path=RESULTS_DIR / "experiment_summary.json",
)

## W&B — syncing offline runs

All runs were saved locally with `mode="offline"`. Two run groups are tracked:
- **`fp32-phase4`** — one run per architecture, FP32 training
- **`qat-phase4`** — one run per architecture, QAT fine-tuning

When ready, sync to the W&B dashboard from a terminal:

```bash
wandb sync --sync-all
```